# 46장. 기기에 맞게 모델 줄이기

『밑바닥부터 만드는 추론 엔진』 46장의 학습을 직접 돌려 보는 노트북입니다.

SmolLM2-135M을 세 가지 방법으로 줄입니다.

- **양자화 인식 학습**: 엔진의 4비트 양자화(`Q4_0`)를 흉내 내면서 학습시킵니다.
- **증류**: 줄인 모델이 원래 모델의 답을 따라 하게 학습시킵니다.
- **가지치기**: 블록을 통째로 덜어 내거나 가중치의 절반을 0으로 만들고 다시 학습시킵니다.

GPU가 있어야 합니다. Colab에서는 `런타임 > 런타임 유형 변경`에서 GPU를
고르십시오. 방법 하나에 몇 분이 걸립니다.

같은 코드로 학습해도 기계와 PyTorch의 판에 따라 가중치가 조금 달라집니다.
책의 숫자는 책의 맥(M4 Max)에서 한 번 돌린 것이고, 여기서 나오는 숫자는
끝자리가 다를 수 있습니다. 품질은 이 노트북이 아니라 **badak으로** 잽니다
(마지막 절).

## 1. 준비

저장소 안에서 실행하면 그대로 쓰고, Colab처럼 저장소가 없는 곳에서는 내려받습니다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("reference/ch46_train.py").exists():
    if Path("../reference/ch46_train.py").exists():
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/chosungmann/building-an-inference-engine-from-scratch"], check=True)
        os.chdir("building-an-inference-engine-from-scratch")
print("작업 디렉터리:", Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "reference/requirements_ch46.txt"], check=False)

## 2. 모델과 글 내려받기

SmolLM2-135M-Instruct(0.27GB)와 WikiText-2를 받아 해시를 확인합니다.

In [ ]:
for script in ("scripts/download_smollm2.sh", "scripts/download_wikitext2.sh"):
    print(subprocess.run(["sh", script], capture_output=True, text=True, check=True).stdout[-600:])

## 3. 흉내 양자화가 하는 일

학습 스크립트에서 가장 중요한 함수를 먼저 봅니다. 가중치를 엔진과 똑같은
방법으로 4비트로 줄였다가 되돌립니다. 값 32개마다 눈금이 하나이고, 값은
열여섯 가지 가운데 하나가 됩니다.

In [ ]:
sys.path.insert(0, "reference")
import torch
from ch46_train import fake_q4_0

torch.manual_seed(0)
weight = torch.randn(1, 32) * 0.05
print("원래 값     ", [round(v, 4) for v in weight[0, :8].tolist()])
print("줄였다 되돌림", [round(v, 4) for v in fake_q4_0(weight)[0, :8].tolist()])
print("서로 다른 값의 수:", fake_q4_0(weight).unique().numel(), "(많아야 16)")

## 4. 학습

방법을 골라 돌립니다. 이름은 `reference/ch46_train.py`의 머리말에 있습니다.
모두 돌리면 30분쯤 걸립니다. 스텝의 수를 줄여 빨리 훑어보려면 `STEPS`를
바꾸십시오(책은 400스텝).

In [ ]:
METHODS = ["ft", "qat", "qat_kd", "pick", "pick_kd", "prune_kd"]
STEPS = "400"

for method in METHODS:
    result = subprocess.run(
        [sys.executable, "reference/ch46_train.py", method],
        capture_output=True, text=True, env={**os.environ, "STEPS": STEPS},
    )
    print("\n".join(line for line in result.stdout.splitlines() if "Loading" not in line))
    if result.returncode != 0:
        print(result.stderr[-2000:])

## 5. 만들어진 모델

방법마다 `models/smollm2_135m_<방법>/`이 생깁니다. badak이 그대로 읽는 폴더입니다.

In [ ]:
for directory in sorted(Path("models").glob("smollm2_135m_*")):
    weights = directory / "model.safetensors"
    if weights.exists():
        print(f"{directory.name:<28} {weights.stat().st_size / 1e6:8.1f}MB")

## 6. badak으로 품질 재기

Rust가 설치된 컴퓨터에서 저장소의 `models/` 아래에 위 폴더들을 두고 다음을
실행합니다. 모델마다 파일에 적힌 그대로의 손실과, 엔진이 4비트로 줄였을
때의 손실을 적습니다.

```bash
cargo run --release -p badak_models --example ch46_quality -- \
    smollm2_135m smollm2_135m_ft smollm2_135m_qat smollm2_135m_qat_kd
```

Colab에서 실행했다면 아래 셀로 폴더 하나를 묶어 내려받을 수 있습니다.

In [ ]:
import shutil

try:
    from google.colab import files

    for method in METHODS:
        archive = shutil.make_archive(f"smollm2_135m_{method}", "zip", "models", f"smollm2_135m_{method}")
        files.download(archive)
except ImportError:
    print("Colab이 아닙니다. 모델은 models/ 아래에 있습니다.")